import librairies et lecture du fichier csv

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import keras
import tensorflow as tf
from keras.models import Sequential
from keras.layers import Dense, Dropout
from keras.optimizers import Adam
from keras.callbacks import EarlyStopping
from keras import layers
from keras.random import shuffle
import nlpaug.augmenter.char as nac
import nlpaug.augmenter.word as naw
import nlpaug.augmenter.word.context_word_embs as nawcwe
import nlpaug.augmenter.word.word_embs as nawwe
import nlpaug.augmenter.word.spelling as naws


In [2]:
import sys
print(sys.executable)

c:\Users\mvm\open3d_vision\.venv\Scripts\python.exe


In [3]:
df = pd.read_csv(r"C:\Users\mvm\open3d_vision\src\layer_raw_and_target_full.csv")

C:\Users\mvm\AppData\Local\Temp\ipykernel_4936\3825118775.py:1: DtypeWarning: Columns (0: owner, 1: layer, 2: linetype, 3: name, 4: insert, 5: elevation, 6: extrusion, 7: pattern_name, 8: start, 9: end, 10: text_direction, 11: style, 12: center, 13: u_pixel, 14: v_pixel, 15: image_size, 16: image_def_handle, 17: image_def_reactor_handle, 18: geometry, 19: defpoint, 20: text_midpoint, 21: text, 22: dimstyle, 23: defpoint2, 24: defpoint3, 25: location, 26: major_axis, 27: align_point, 28: uid, 29: style_handle, 30: text_style_handle, 31: block_scale_vector, 32: color_name, 33: horizontal_direction, 34: defpoint4, 35: defpoint5, 36: normal_vector, 37: leader_offset_annotation_placement, 38: annotation_handle, 39: vtx0, 40: vtx1, 41: vtx2, 42: vtx3, 43: prompt, 44: tag, 45: underlay_def_handle, 46: arrow_head_handle, 47: bg_fill_color_name, 48: start_tangent, 49: end_tangent, 50: unit_vector) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(r"C:\

Exploration sommaire de la table

In [4]:
df.head()


,file_name,entity_type,handle,owner,layer,linetype,lineweight,color,name,insert,...,default_end_width,thickness,fit_tolerance,bg_fill_color_name,paperspace,start_tangent,end_tangent,unit_vector,target,target_nom
0,034_diek_aut_fac_2024.06.06.dxf,lwpolyline,e5c7,1f,NaN,0,NaN,114.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,autre
1,034_diek_aut_fac_2024.06.06.dxf,lwpolyline,e654,1f,NaN,0,NaN,114.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,autre
2,034_diek_aut_fac_2024.06.06.dxf,mtext,e690,1f,NaN,NaN,NaN,114.0,NaN,"(453.1794274376368, 26.83150764608996, 0.0)",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,autre
3,034_diek_aut_fac_2024.06.06.dxf,lwpolyline,e6cb,1f,NaN,0,NaN,114.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,autre
4,034_diek_aut_fac_2024.06.06.dxf,mtext,e6ce,1f,NaN,NaN,NaN,114.0,NaN,"(453.1794274376368, 26.83150764608996, 0.0)",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,autre


Erreur à régler: le layer "cotation mur" n'apparaît nul part, et s'il devait être rangé dans un autre layer ce serait "cotation" et pas "mur porteur"

In [ ]:
df.drop(["file_name","pattern_type","pattern_angle","pattern_scale","pattern_double","flags","true_color","attachment_point","text_direction","flow_direction","actual_measurement","line_spacing_style","line_spacing_factor","char_height","width","defined_height","rotation"],axis=1,inplace=True)


In [6]:
raw_csv = df.to_numpy()

In [7]:
print(raw_csv.ndim)
print(raw_csv.shape)
print(raw_csv.size)
print(raw_csv.itemsize)
print(raw_csv.nbytes)


2
(8269883, 123)
1017195609
8
8137564872


In [8]:
df["target_nom"].unique()

<ArrowStringArray>
[             'autre',          'TERRASSES',        'MUR PORTEUR',
           'CLOISONS',             'PORTES',           'FENETRES',
          'ESCALIERS',           'HACHURES',           'COTATION',
              'COUPE',              'TEXTE',            'SURFACE',
 'LIMITE PARCELLAIRE',            'PARKING']
Length: 14, dtype: str

ne pas tracer = "coupe", "terrasses", "cotation", "surface", "texte", "parking", "limite parcellaire", "hachures"

tracer comme un mur = "cloisons", "mur porteur", "escaliers", "portes"

In [9]:
import numpy as np
import nlpaug.augmenter.word as naw
from gensim.models import KeyedVectors
# Contourner le bug nlpaug : glove.py utilise KeyedVectors sans l'importer
import nlpaug.model.word_embs.glove as _glove_module
_glove_module.KeyedVectors = KeyedVectors

In [10]:
# Filtrer les lignes où la dernière colonne ne vaut pas "autre"
target_csv = raw_csv[raw_csv[:, -1] != "autre"]

Création d'un échantillon formé aléatoirement

In [11]:
np.random.shuffle(target_csv)
echantillon_csv = target_csv[:2000]
unique_values =[0] * 14
print(unique_values)
for value in echantillon_csv[:,-2]:
    unique_values[value-1] += 1
print("modalités parsées:",unique_values)

[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
modalités parsées: [5, 851, 188, 85, 91, 62, 70, 214, 0, 109, 120, 56, 123, 26]


Enlevons la colonne 2, il y a quasiment autant de valeurs différentes que de lignes (aucune info exploitable)

In [12]:
echantillon_csv = np.delete(echantillon_csv,1,1)
echantillon_csv

array([['lwpolyline', 1450000000000.0, 'coupe', ..., nan, 10, 'COUPE'],
       ['line', '1f', '000 plans coupes', ..., nan, 10, 'COUPE'],
       ['text', '55890', 'texte-lot', ..., nan, 11, 'TEXTE'],
       ...,
       ['lwpolyline', '1f',
        'ar_b_np                                                          mur béton non porteur',
        ..., nan, 2, 'MUR PORTEUR'],
       ['line', 445334, 'cloisons', ..., nan, 3, 'CLOISONS'],
       ['line', '1f', '0-schraffuren', ..., nan, 7, 'HACHURES']],
      shape=(2000, 122), dtype=object)

Certaines colonnes ont des types variables, des tuples enregistrés comme des str par exemple

In [13]:
# définition de fonction: s'assurer que toutes les lignes d'une colonne ont le même type
def unifier_types_colonnes(arr, priorite=('float', 'int', 'tuple', 'list', 'str')):
    """
    Unifie le type de chaque colonne en priorisant numériques (float, int),
    puis tuple/liste, puis str. Retourne une copie du tableau.
    """
    import ast
    import numpy as np
    import pandas as pd
    arr = np.asarray(arr, dtype=object)
    n_rows, n_cols = arr.shape
    out = np.empty_like(arr)

    def _is_na(x):
        if x is None:
            return True
        if isinstance(x, float) and np.isnan(x):
            return True
        try:
            if pd.isna(x):
                return True
        except (TypeError, ValueError):
            pass
        return False

    def _parse_literal(s):
        """Si s est une str représentant un tuple/liste (ex. '(1, 2, 0)'), retourne le type Python ; sinon None."""
        if not isinstance(s, str) or not s.strip():
            return None
        s = s.strip()
        try:
            v = ast.literal_eval(s)
            if isinstance(v, tuple):
                return 'tuple', v
            if isinstance(v, list):
                return 'list', v
        except (ValueError, SyntaxError):
            pass
        return None

    # Fonction d'attribution d'un nom de type à une valeur, selon la priorité attendue pour l'homogénéisation des colonnes.
    # Les chaînes du type "(x, y, z)" sont considérées comme tuple pour la détection.
    def _type_priorite(x):
        if _is_na(x):
            return None
        if isinstance(x, (int, float)) and not isinstance(x, bool):
            return 'float' if isinstance(x, float) else 'int'
        if isinstance(x, tuple):
            return 'tuple'
        if isinstance(x, list):
            return 'list'
        if isinstance(x, str):
            parsed = _parse_literal(x)
            if parsed is not None:
                return parsed[0]
            return 'str'
        return type(x).__name__

    # Boucle principale pour chaque colonne : on collecte les types présents, choisit celui à garder
    # selon la priorité, puis on convertit/force les valeurs colonne par colonne.
    for j in range(n_cols):
        col = arr[:, j]
        types_presents = {}
        # On compte les occurrences de chaque type non-na dans la colonne
        for x in col:
            t = _type_priorite(x)
            if t is not None:
                types_presents[t] = types_presents.get(t, 0) + 1
        # Si aucun type détecté (que des NA) : colonne à NaN
        if not types_presents:
            out[:, j] = np.nan
            continue
        # On sélectionne le type à utiliser pour homogénéiser, selon la priorité définie
        choix = None
        for p in priorite:
            if p in types_presents:
                choix = p
                break
        if choix is None:
            choix = next(iter(types_presents.keys()))
        # Conversion de chaque valeur de la colonne selon le type choisi
        for i in range(n_rows):
            x = col[i]
            if _is_na(x):
                out[i, j] = np.nan if choix in ('float', 'int') else '' if choix == 'str' else x
                continue
            if choix in ('float', 'int'):
                try:
                    out[i, j] = float(x)
                except (TypeError, ValueError):
                    out[i, j] = np.nan
            elif choix == 'str':
                out[i, j] = str(x)
            elif choix == 'tuple':
                parsed = _parse_literal(x) if isinstance(x, str) else None
                if parsed is not None:
                    out[i, j] = tuple(parsed[1]) if isinstance(parsed[1], list) else parsed[1]
                elif isinstance(x, list):
                    out[i, j] = tuple(x)
                else:
                    out[i, j] = x
            elif choix == 'list':
                parsed = _parse_literal(x) if isinstance(x, str) else None
                if parsed is not None:
                    out[i, j] = list(parsed[1]) if isinstance(parsed[1], tuple) else parsed[1]
                elif isinstance(x, tuple):
                    out[i, j] = list(x)
                else:
                    out[i, j] = x
            else:
                out[i, j] = x
    return out

# Fonction utilitaire de debug : affiche pour chaque colonne le nombre de types distincts rencontrés
# et combien de valeurs de chaque type. Optionnellement, vise une colonne précise (col_index)
def afficher_types_par_colonne(arr, col_index=None):
    """Debug : affiche le nombre de types présents par colonne (ou pour la colonne col_index)."""
    import ast
    import numpy as np
    import pandas as pd
    arr = np.asarray(arr, dtype=object)
    n_cols = arr.shape[1]

    def _parse_literal(s):
        if not isinstance(s, str) or not s.strip():
            return None
        try:
            v = ast.literal_eval(s.strip())
            return type(v).__name__ if isinstance(v, (tuple, list)) else None
        except (ValueError, SyntaxError):
            return None

    # Fonction qui donne le nom du type lisible pour une valeur donnée, y compris NA
    # Les str du type "(x, y, z)" sont affichées comme 'tuple'.
    def _type_name(x):
        if x is None or (isinstance(x, float) and np.isnan(x)):
            return 'NA'
        try:
            if pd.isna(x):
                return 'NA'
        except (TypeError, ValueError):
            pass
        if isinstance(x, bool):
            return 'bool'
        if isinstance(x, int):
            return 'int'
        if isinstance(x, float):
            return 'float'
        if isinstance(x, str):
            t = _parse_literal(x)
            return t if t is not None else 'str'
        if isinstance(x, tuple):
            return 'tuple'
        if isinstance(x, list):
            return 'list'
        return type(x).__name__

    indices = [col_index] if col_index is not None else range(n_cols)
    for j in indices:
        col = arr[:, j]
        counts = {}
        for x in col:
            t = _type_name(x)
            counts[t] = counts.get(t, 0) + 1
        nb_types = len(counts)
        print(f"Colonne {j} : {nb_types} type(s) → {dict(sorted(counts.items(), key=lambda e: -e[1]))}")

def colonnes_nan_only(arr, supprimer=False):
    """
    Affiche les colonnes ne contenant que des NaN (type) ou la chaîne 'nan'.
    Si supprimer=True, retourne (arr sans ces colonnes, indices_supprimés). Sinon retourne (arr, indices).
    """
    import numpy as np
    import pandas as pd
    arr = np.asarray(arr, dtype=object)
    n_rows, n_cols = arr.shape

    def _est_nan_like(x):
        if x is None:
            return True
        if isinstance(x, float) and np.isnan(x):
            return True
        try:
            if pd.isna(x):
                return True
        except (TypeError, ValueError):
            pass
        if isinstance(x, str) and x.strip().lower() == 'nan':
            return True
        return False

    indices_nan_only = []
    for j in range(n_cols):
        col = arr[:, j]
        if all(_est_nan_like(x) for x in col):
            indices_nan_only.append(j)

    print(f"Colonnes ne contenant que des NaN (type ou str 'nan') : {len(indices_nan_only)} → {indices_nan_only}")

    if supprimer and indices_nan_only:
        mask = np.ones(n_cols, dtype=bool)
        mask[indices_nan_only] = False
        out = arr[:, mask]
        print(f"Colonnes supprimées. Nouvelle forme : {out.shape}")
        return out, indices_nan_only
    return arr, indices_nan_only



In [14]:
echantillon_csv = unifier_types_colonnes(echantillon_csv)
afficher_types_par_colonne(echantillon_csv)

Colonne 0 : 1 type(s) → {'str': 2000}
Colonne 1 : 2 type(s) → {'NA': 1453, 'float': 547}
Colonne 2 : 1 type(s) → {'str': 2000}
Colonne 3 : 1 type(s) → {'str': 2000}
Colonne 4 : 2 type(s) → {'float': 1249, 'NA': 751}
Colonne 5 : 2 type(s) → {'float': 1140, 'NA': 860}
Colonne 6 : 1 type(s) → {'str': 2000}
Colonne 7 : 2 type(s) → {'NA': 1732, 'tuple': 268}
Colonne 8 : 2 type(s) → {'NA': 1827, 'tuple': 173}
Colonne 9 : 2 type(s) → {'NA': 1725, 'tuple': 275}
Colonne 10 : 1 type(s) → {'str': 2000}
Colonne 11 : 2 type(s) → {'NA': 1830, 'float': 170}
Colonne 12 : 2 type(s) → {'NA': 1830, 'float': 170}
Colonne 13 : 2 type(s) → {'NA': 1830, 'float': 170}
Colonne 14 : 2 type(s) → {'NA': 1854, 'float': 146}
Colonne 15 : 2 type(s) → {'tuple': 1247, 'NA': 753}
Colonne 16 : 2 type(s) → {'tuple': 1247, 'NA': 753}
Colonne 17 : 1 type(s) → {'str': 2000}
Colonne 18 : 2 type(s) → {'NA': 1883, 'tuple': 117}
Colonne 19 : 2 type(s) → {'NA': 1888, 'float': 112}
Colonne 20 : 2 type(s) → {'NA': 1904, 'float': 9

In [15]:
echantillon_csv, inutile = colonnes_nan_only(echantillon_csv, supprimer = True)

Colonnes ne contenant que des NaN (type ou str 'nan') : 55 → [47, 60, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119]
Colonnes supprimées. Nouvelle forme : (2000, 67)


In [16]:
#définition de fonctions: debug sur les types de colonnes présentes
def afficher_valeurs_uniques(arr):
    """
    Affiche toutes les valeurs uniques d'un array numpy (1D ou 2D).
    Gère les colonnes en dtype=object avec types mélangés (int, float, str, None, etc.).
    Les chaînes du type "(x, y, z)" sont affichées comme type tuple.
    """
    import ast
    arr = np.asarray(arr)
    flat = arr.ravel()

    def _is_na(x):
        if x is None:
            return True
        if isinstance(x, float) and np.isnan(x):
            return True
        try:
            if pd.isna(x):
                return True
        except (TypeError, ValueError):
            pass
        return False

    def _display_type(x):
        if isinstance(x, str) and x.strip():
            try:
                v = ast.literal_eval(x.strip())
                if isinstance(v, (tuple, list)):
                    return type(v).__name__
            except (ValueError, SyntaxError):
                pass
        return type(x).__name__

    def _norm_key(x):
        """Clé pour tri et déduplication : (type, repr) pour éviter mélange 1 vs '1'."""
        if _is_na(x):
            return (type(None).__name__, "<NA>")
        return (type(x).__name__, repr(x))

    seen = {}
    for x in flat:
        key = _norm_key(x)
        if key not in seen:
            seen[key] = x

    uniques = list(seen.values())
    uniques.sort(key=_norm_key)

    print(f"Nombre de valeurs uniques : {len(uniques)}")
    if len(uniques) <= 100:
        for i, v in enumerate(uniques):
            na = " (NA)" if _is_na(v) else ""
            print(f"  [{i}] {repr(v)}{na}  — type: {_display_type(v)}")
    else:
        for i, v in enumerate(uniques[:50]):
            na = " (NA)" if _is_na(v) else ""
            print(f"  [{i}] {repr(v)}{na}  — type: {_display_type(v)}")
        print(f"  ... et {len(uniques) - 50} autres (total {len(uniques)})")
    return uniques

In [17]:
# définition de fonctions: Nombre de valeurs uniques par colonne, ordre décroissant
def _is_na(x):
    if x is None:
        return True
    if isinstance(x, float) and np.isnan(x):
        return True
    try:
        if pd.isna(x):
            return True
    except (TypeError, ValueError):
        pass
    return False

def _norm_key(x):
    if _is_na(x):
        return (type(None).__name__, "<NA>")
    return (type(x).__name__, repr(x))

def compter_uniques(col):
    seen = {}
    for x in col:
        key = _norm_key(x)
        if key not in seen:
            seen[key] = x
    return len(seen), list(seen.values())





In [18]:
#debug: affichage des colonnes en fonction de leur modalités, avec exemple des 5 premières occurences
n_cols = echantillon_csv.shape[1]
counts = []
for j in range(n_cols):
    n, uniques = compter_uniques(echantillon_csv[:, j])
    counts.append((j, n, uniques))

counts.sort(key=lambda t: t[1], reverse=True)
print(f"Colonnes : {n_cols}, Lignes : {len(echantillon_csv)}")
print("Nombre de valeurs uniques par colonne (ordre décroissant) :")
for j, n, uniques in counts:
    print(f"  Colonne {j} : {n} unique(s)")
    to_show = uniques[:5]
    for i, v in enumerate(to_show):
        na = " (NA)" if _is_na(v) else ""
        print(f"    [{i}] {repr(v)}{na} — type: {type(v).__name__}")
    if len(uniques) > 5:
        print(f"    ... et {len(uniques) - 5} autres")

Colonnes : 67, Lignes : 2000
Nombre de valeurs uniques par colonne (ordre décroissant) :
  Colonne 16 : 1247 unique(s)
    [0] nan (NA) — type: float
    [1] (1013586.859292127, -33533.6885326882, 0.0) — type: tuple
    [2] (1.115170532226563, 0.2194087524414063, 0.0) — type: tuple
    [3] (100250.1372168582, 83031.68354824434, 0.0) — type: tuple
    [4] (0.4340709838867188, 0.3270087280273438, 0.0) — type: tuple
    ... et 1242 autres
  Colonne 15 : 1246 unique(s)
    [0] nan (NA) — type: float
    [1] (1013588.514077719, -33533.36430135338, 0.0) — type: tuple
    [2] (1.11288427734375, 0.2196602172851563, 0.0) — type: tuple
    [3] (100250.0782406155, 83031.59136414687, 0.0) — type: tuple
    [4] (0.4550703125, 0.327006591796875, 0.0) — type: tuple
    ... et 1241 autres
  Colonne 2 : 309 unique(s)
    [0] 'coupe' — type: str
    [1] '000 plans coupes' — type: str
    [2] 'texte-lot' — type: str
    [3] 'cloison briques' — type: str
    [4] '0-schraffuren' — type: str
    ... et 304 

In [19]:
import numpy as np
import pandas as pd

_F32_MAX = np.finfo(np.float32).max

def _to_float_oh(val):
    """
    Convertit en float utilisable en float32 : clip les très grandes valeurs (coordonnées DXF)
    pour éviter overflow → inf → loss NaN en entraînement.
    """
    if val is None or (isinstance(val, float) and np.isnan(val)) or (isinstance(val, str) and str(val).strip() == ''):
        return 0.0
    try:
        x = np.float64(val)
        if not np.isfinite(x):
            return 0.0
        x = np.clip(x, -_F32_MAX, _F32_MAX)
        return float(np.float32(x))
    except (TypeError, ValueError, OverflowError):
        return 0.0

def one_hot_encode_echantillon(data, categorical_col_indices=None, dtype=np.float32):
    """
    Encode les colonnes catégorielles en one-hot. Les autres colonnes sont converties en float32.
    Prêt pour Keras.
    """
    data = np.asarray(data)
    n_rows, n_cols = data.shape

    #création de la taille de l'array de one-hot si on ne le connaît pas à l'avance
    if categorical_col_indices is None:
        categorical_col_indices = []
        for j in range(n_cols):
            col = data[:, j]
            # ~ sert à inverser un tableau de booléens 
            non_null = col[~pd.isna(col)]
            if len(non_null) > 0:
                n_str = sum(1 for x in non_null if isinstance(x, str))
                if n_str > len(non_null) / 2:
                    categorical_col_indices.append(j)

    mappings = {}
    parts = []

    #pour chaque colonne
    for j in range(n_cols):
        #extraire la colonne en une variable
        col = data[:, j]
        #si la colonne actuelle fait partie des colonnes catégorielles
        if j in categorical_col_indices:
            #extraire les valeurs uniques de la colonne
            uniques = sorted(set(str(x) for x in col if pd.notna(x) and str(x).strip() != ''))
            #s'il n'y a pas de valeurs uniques, les valeurs sont à nulles
            if not uniques:
                uniques = ['__nan__']
            #complète le dictionnaire, avec l'indice de la valeur unique ainsi qu'elle-même
            mappings[j] = {v: i for i, v in enumerate(uniques)}
            #nombre de valeurs uniques
            n_cats = len(uniques)
            #création d'un tableau de zéros, de la taille de la colonne, et de la taille de la ligne
            oh = np.zeros((n_rows, n_cats), dtype=dtype)
            #pour chaque ligne
            for i in range(n_rows):
                #v = valeur de la colonne
                v = col[i]
                #vérification de la valeur de v
                key = '__nan__' if (pd.isna(v) or str(v).strip() == '') else str(v)
                #récupération de l'indice de la valeur unique
                idx = mappings[j].get(key, 0)
                #on met à 1 la valeur unique
                oh[i, idx] = 1.0
            #ajout du tableau
            parts.append(oh)
        else:
            col_f = np.array([_to_float_oh(col[i]) for i in range(n_rows)], dtype=dtype)
            parts.append(col_f.reshape(-1, 1))

    return np.hstack(parts).astype(dtype), mappings

In [20]:
COL_IDX = 2  # quatrième colonne (index 0 = 1ère colonne)
# Sélectionner toutes les lignes où la 4ème colonne n'est pas NaN (masque numpy pour indexation)
texts = [str(x).strip() for x in echantillon_csv[:,2]]

In [21]:
# Data augmentation sur TOUTES les lignes où la 4ème colonne n'est pas NaN
MODEL_PATH = r"C:\Users\mvm\open3d_vision\data\glove.6B.300d.txt"
aug_w2v = naw.WordEmbsAug(model_type="glove", model_path=MODEL_PATH, action="substitute", aug_p=0.2)
augmented_texts = aug_w2v.augment(texts, num_thread=8)

In [22]:
# Ajout des lignes augmentées à echantillon_csv si différentes de l'original.
# On suppose que chaque ligne a toujours un layer (pas besoin de vérifier la validité des indices)
augmented_rows = []

for idx, aug_text in enumerate(augmented_texts):
    # nlpaug peut renvoyer une liste (1er candidat) ou None en cas d'échec
    if isinstance(aug_text, list):
        aug_text = aug_text[0] if aug_text else None
    if aug_text is None:
        continue
    aug_str = str(aug_text).strip()
    original_text = str(echantillon_csv[idx, COL_IDX]).strip()
    if aug_str == original_text:
        continue
    # Copie de la ligne (même type object), seule la colonne texte change
    new_row = np.array(echantillon_csv[idx].copy(), dtype=object, copy=True)
    new_row[COL_IDX] = aug_str
    augmented_rows.append(new_row)

if augmented_rows:
    echantillon_csv = np.vstack([echantillon_csv, *augmented_rows])

print(f"Word embedding : {len(augmented_rows)} textes augmentés ajoutés à echantillon_csv (colonne {COL_IDX + 1}). TAILLE FINALE : {len(echantillon_csv)} lignes.")

Word embedding : 1329 textes augmentés ajoutés à echantillon_csv (colonne 3). TAILLE FINALE : 3329 lignes.


Encodage multi-hot des valeurs non numériques

Séparation du target du dataset

In [23]:
# Extraction des deux dernières colonnes de echantillon_csv
deux_dernieres_colonnes = np.zeros((1,2))
if(deux_dernieres_colonnes.shape[1] == 2):
    deux_dernieres_colonnes = echantillon_csv[:, -2:]
    echantillon_csv = echantillon_csv[:, :-2]
deux_dernieres_colonnes

array([[10.0, 'COUPE'],
       [10.0, 'COUPE'],
       [11.0, 'TEXTE'],
       ...,
       [3.0, 'CLOISONS'],
       [2.0, 'MUR PORTEUR'],
       [7.0, 'HACHURES']], shape=(3329, 2), dtype=object)

ne pas tracer = "coupe", "terrasses", "cotation", "surface", "texte", "parking", "limite parcellaire", "hachures"

tracer comme un mur = "cloisons", "mur porteur", "escaliers", "portes"

In [24]:
# Première colonne = 0 si 2e colonne dans "ne pas tracer", sinon 1
ne_pas_tracer = {"coupe", "terrasses", "cotation", "surface", "texte", "parking", "limite parcellaire", "hachures"}
for i in range(len(deux_dernieres_colonnes)):
    label = str(deux_dernieres_colonnes[i, 1]).strip().lower()
    deux_dernieres_colonnes[i, 0] = 0 if label in ne_pas_tracer else 1



In [25]:
from keras import layers
import numpy as np
copie_echantillon = np.copy(echantillon_csv)
#définition du modèle d'encoding multi-hot
max_tokens = 20_000  # 20_000 is more readable
text_vectorization = layers.TextVectorization(
    max_tokens=max_tokens,
    split="whitespace",
    output_mode="multi_hot",
)

# Adaptation du vectorizer sur la colonne 
text_vectorization.adapt(copie_echantillon[:, COL_IDX])

# Affichage du dictionnaire pour vérification
print("Vocabulaire vectorizer :", text_vectorization.get_vocabulary()[:50], "... (total:", len(text_vectorization.get_vocabulary()), "mots)")


# Encodage multi_hot de toute la colonne 
col4_encoded_multi_hot = text_vectorization(copie_echantillon[:, COL_IDX])
for value in range(0,len(col4_encoded_multi_hot)):
    copie_echantillon[value, COL_IDX] = col4_encoded_multi_hot[value]
# Affichage de vérification pour les 3 premières lignes
print("Encodage multi_hot des 3 premières valeurs de la colonne 4 :")
print(np.array(col4_encoded_multi_hot[0]))

Vocabulaire vectorizer : ['[UNK]', np.str_('porteur'), np.str_('mur'), np.str_('wand'), np.str_('cotation'), np.str_('aussen'), np.str_('a'), np.str_('armisol'), np.str_('cloisons'), np.str_('murs'), np.str_('wall'), np.str_('isolation'), np.str_('intérieure'), np.str_('awall'), np.str_('neubau10'), np.str_('07menuiserie'), np.str_('béton'), np.str_('ext'), np.str_('de'), np.str_('coupe'), np.str_('außen'), np.str_('2d'), np.str_('avp'), np.str_('arasec'), np.str_('ar1'), np.str_('portes'), np.str_('à'), np.str_('extérieurs'), np.str_('sec'), np.str_('arbpo'), np.str_('cotes'), np.str_('awallpatt'), np.str_('iwall'), np.str_('briques'), np.str_('cloison'), np.str_('fenetre'), np.str_('hachures'), np.str_('int'), np.str_('fenetres'), np.str_('non'), np.str_('extérieursaa'), np.str_('séparation'), np.str_('pussep'), np.str_('escaliers'), np.str_('bpa6mur'), np.str_('parking'), np.str_('163texteniveau'), np.str_('021limitenumerique'), np.str_('schraffuren'), np.str_('patt')] ... (total: 1

In [26]:
# Nombre de colonnes à valeurs catégorielles / string dans echantillon_csv
n_cols = copie_echantillon.shape[1]
string_col_indices = []
for j in range(n_cols):
    #prendre la colonne courante
    col = copie_echantillon[:, j]
    #tester si la colonne est vide ou non
    non_null = col[~pd.isna(col)]
    if len(non_null) == 0:
        continue

    n_str = sum(1 for x in non_null if isinstance(x, str))
    if n_str > len(non_null) / 2:
        string_col_indices.append(j)

print(f"Nombre de colonnes à valeurs catégorielles (string) : {len(string_col_indices)} sur {n_cols}")
print(f"Indices de ces colonnes : {string_col_indices}")

Nombre de colonnes à valeurs catégorielles (string) : 8 sur 65
Indices de ces colonnes : [0, 3, 6, 10, 17, 37, 43, 64]


In [27]:

n_features = echantillon_csv.shape[1]
slice_ = echantillon_csv[:2000, n_features-68][1000:1010]
has_not_nan = any([not pd.isna(x) for x in slice_])
print("Présence d'au moins un élément non-NaN dans le slice :", has_not_nan)

Présence d'au moins un élément non-NaN dans le slice : False


TO FIX:

In [34]:
# Matrice numérique : echantillon_csv contient encore des str ('line', etc.) → asarray(..., float32) échoue.
# La colonne COL_IDX de copie_echantillon contient les vecteurs multi-hot (TF/NumPy) ; le reste passe par one_hot_encode_echantillon.

def _as_float_vector(v):
    x = v.numpy() if hasattr(v, "numpy") else v
    return np.asarray(x, dtype=np.float32).ravel()

text_matrix = np.stack([_as_float_vector(v) for v in copie_echantillon[:, COL_IDX]])
n_cols_ce = copie_echantillon.shape[1]
other_idx = [j for j in range(n_cols_ce) if j != COL_IDX]
other_data = copie_echantillon[:, other_idx]
old_to_new = {old: new for new, old in enumerate(other_idx)}
string_cols_other = [old_to_new[j] for j in string_col_indices if j != COL_IDX]

X_rest, _ = one_hot_encode_echantillon(other_data, categorical_col_indices=string_cols_other)
X_all = np.hstack([X_rest, text_matrix]).astype(np.float32)
X_all = np.nan_to_num(X_all, nan=0.0, posinf=0.0, neginf=0.0)
y_all = np.asarray(deux_dernieres_colonnes[:, 0], dtype=np.float32)

n_samples = X_all.shape[0]
rng = np.random.default_rng()
perm = rng.permutation(n_samples)
X_all = X_all[perm]
y_all = y_all[perm]

n_train, n_val = 2000, 1000
train_values = X_all[:n_train]
train_target = y_all[:n_train]
validation_values = X_all[n_train : n_train + n_val]
validation_target = y_all[n_train : n_train + n_val]
test_values = X_all[n_train + n_val :]
test_target = y_all[n_train + n_val :]

n_features = X_all.shape[1]
print(
    f"X: {X_all.shape}, y: {y_all.shape} | "
    f"train {train_values.shape}, val {validation_values.shape}, test {test_values.shape}"
)


X: (3329, 2037), y: (3329,) | train (2000, 2037), val (1000, 2037), test (329, 2037)


C:\Users\mvm\AppData\Local\Temp\ipykernel_4936\4073522358.py:68: RuntimeWarning: overflow encountered in cast
  col_f = np.array([_to_float_oh(col[i]) for i in range(n_rows)], dtype=dtype)


Création du modèle

In [41]:
model = keras.Sequential()
model.add(keras.Input(shape=(n_features,)))
model.add(Dense(150, activation='relu'))
model.add(Dense(150, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(1, activation='sigmoid'))
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
model.fit(train_values, train_target, epochs=250, batch_size=64, validation_data=(validation_values, validation_target))

Epoch 1/250
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.5870 - loss: 479551552.0000 - val_accuracy: 0.6800 - val_loss: 78881536.0000
Epoch 2/250
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6895 - loss: 364243488.0000 - val_accuracy: 0.7240 - val_loss: 330807936.0000
Epoch 3/250
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7260 - loss: 235321280.0000 - val_accuracy: 0.7500 - val_loss: 43236132.0000
Epoch 4/250
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7835 - loss: 192387200.0000 - val_accuracy: 0.7990 - val_loss: 88524352.0000
Epoch 5/250
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8410 - loss: 128780488.0000 - val_accuracy: 0.8520 - val_loss: 142681408.0000
Epoch 6/250
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8665 - loss: 75122136.0000 - val_accuracy: 0.8610 - val_loss: 21108888.0000
Epoch 7/250
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8770 - loss: 54572340.0000 - val_accuracy: 0.8560 - val_loss: 19352480.0000
Epoch 

In [43]:
# Sauvegarde du modèle tel quel (architecture, poids, config de compilation)
MODEL_SAVE_PATH = r"C:\Users\mvm\open3d_vision\src\layer_dwg_classifier_model.keras"
model.save(MODEL_SAVE_PATH)
print(f"Modèle enregistré : {MODEL_SAVE_PATH}")

Modèle enregistré : C:\Users\mvm\open3d_vision\src\layer_dwg_classifier_model.keras


In [42]:
model.evaluate(test_values, test_target)

11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9362 - loss: 0.1728 


[0.17279057204723358, 0.936170220375061]